In [ ]:
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from time import time

from Configuration import CLASSIFIER_CONFIGURATION, TRANSFORMER_CONFIGURATION, PATH_CONFIGURATION
from Tokenizers.tokenizer import BasePoetryTokenizer, SyllableTokenizer, WordPieceTokenizer
from models.Transformers import PoetEmbedder
from models.Pooling import AttentionPoemClassifier
from datasets.RobustAdversarialDataset import RobustAdversarialDataset


POETRY_PATH = PATH_CONFIGURATION.DATASETS_PATH()['POETRY']
NON_POETRY_PATH = PATH_CONFIGURATION.DATASETS_PATH()['NON-POETRY']
SYL_CLASSIFIER_WEIGHTS_PATH = PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()['SYLLABLE_CLASSIFIER']
WORD_CLASSIFIER_WEIGHTS_PATH = PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()['WORDPIECE_CLASSIFIER']
SYLLABLE_TRANSFORMER_PATH = PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()['SYLLABLE_TRANSFORMER']
WORDPIECE_TRANSFORMER_PATH = PATH_CONFIGURATION.MODELS_WEIGHTS_PATH()['WORDPIECE_TRANSFORMER']

EMBEDDING_DIM = CLASSIFIER_CONFIGURATION.PARAMETERS()['HIDDEN_DIM']
DROPOUT = CLASSIFIER_CONFIGURATION.PARAMETERS()['DROPOUT']
MAX_SEQ_LEN = CLASSIFIER_CONFIGURATION.PARAMETERS()['MAX_SEQ_LEN']

BATCH_SIZE = CLASSIFIER_CONFIGURATION.TRAINING_PARAMETERS()['BATCH_SIZE']
MAX_LR = CLASSIFIER_CONFIGURATION.TRAINING_PARAMETERS()['LEARNING_RATE']
EPOCHS = CLASSIFIER_CONFIGURATION.TRAINING_PARAMETERS()['EPOCHS']
SEED = CLASSIFIER_CONFIGURATION.TRAINING_PARAMETERS()['SEED']

# Scelta dei layer
SYL_LAYERS = CLASSIFIER_CONFIGURATION.PARAMETERS()['SYLLABLE_LAYERS']
WP_LAYERS = CLASSIFIER_CONFIGURATION.PARAMETERS()['WORDPIECE_LAYERS']

random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

true_poems_raw = BasePoetryTokenizer.get_all_poems_from_directory(POETRY_PATH)
fake_poems_raw = BasePoetryTokenizer.get_all_poems_from_directory(NON_POETRY_PATH)
word_tok = WordPieceTokenizer.from_config()
syl_tok = SyllableTokenizer.from_config()

print(f"Vocabulary size (WordPiece): {len(word_tok.vocab)}")
print(f"Vocabulary size (Syllable): {len(syl_tok.vocab)}")

Vocabulary size (WordPiece): 15000
Vocabulary size (Syllable): 16561


In [ ]:
import random

from datasets.PoetryCorruptionUtils import WordAndSyllableCorruptionMethod, controlTokenGeneration, dropControlTokens, nonLinearLabelling, parallelized_corruption_generation, sequential_corruption_generation

def generate_dataset(
    true_poems_raw,
    fake_poems_raw,
    word_tok,
    syl_tok,
    corruption_rates,

    # ---- Parametri di bilanciamento globale ----
    max_total_samples=15000,           # Limite sul dataset congiunto (train+test)
    hard_negatives_target_pct=0.20,     # 20% reale di target 0.000 sul totale globale
    intact_poems_pct=0.12,              # 12% poesie integre (target 1.0)
    intact_paraphrases_pct=0.15,        # 15% parafrasi integre (target 0.25)
    
    # ---- Generazione dei Negative Hard ----
    num_nonsense_negatives=1000,       # Sequenze casuali OOD (es. token ripetuti / spazzatura)

    # ---- Etichette ----
    poem_rate=2.0,
    poem_ceil=1.0,
    poem_floor=0.0,
    paraphrase_rate=2.0,
    paraphrase_ceil=0.25,
    paraphrase_floor=0.0,

    # ---- Augmentation ----
    control_dropout_prob=0.3,
    max_control_tokens_per_sequence=12,

    # ---- Altro ----
    num_workers=0,
    train_split_ratio=0.9,
    seed=42,
):
    """
    Genera un dataset bilanciato mantenendo la dimensione globale controllata (~15k campioni)
    e garantendo una presenza reale del 20% di Hard Negatives (target 0.0) equamente divisi tra Train e Test.
    
    Tutti i campioni (Poesie, Parafrasi, Hard Negatives) vengono dotati del tag [VERSE]
    per costringere il modello a valutare il contenuto semantico/ritmico anziché il semplice control token.
    """
    if seed is not None:
        random.seed(seed)

    assert 0.0 <= control_dropout_prob <= 1.0
    rng = random.Random(seed)

    n_poems_avail = len(true_poems_raw)
    n_paraphrases_avail = len(fake_poems_raw)

    # ---------------- 1. CALCOLO QUOTE ASSOLUTE SUL TOTALE ----------------
    n_hard_negatives_total = int(max_total_samples * hard_negatives_target_pct)   # ~3.000 (target 0.0)
    n_true_poems_intact = int(max_total_samples * intact_poems_pct)               # ~1.800 (target 1.0)
    n_paraphrases_used = int(max_total_samples * intact_paraphrases_pct)             # ~2.250 (target 0.25)
    
    # Allocazione rimanente per le corruzioni multilivello (~53% dei campioni)
    n_corrupted_samples_total = max_total_samples - (n_hard_negatives_total + n_true_poems_intact + n_paraphrases_used)
    
    active_rates = [r for r in corruption_rates if r > 0.0]
    n_active_rates = max(len(active_rates), 1)

    # Distribuzione 75% poesie e 25% parafrasi per le corruzioni
    n_corrupted_poems_total = int(n_corrupted_samples_total * 0.75)
    n_corrupted_paraphrases_total = n_corrupted_samples_total - n_corrupted_poems_total

    n_corrupted_poems_per_rate = n_corrupted_poems_total // n_active_rates
    n_corrupted_paraphrases_per_rate = n_corrupted_paraphrases_total // n_active_rates

    # ---------------- 2. CAMPIONAMENTO POOL DA RAM ----------------
    n_poems_needed = min(n_true_poems_intact + n_corrupted_poems_total, n_poems_avail)
    n_paraphrases_needed = min(n_paraphrases_used + n_corrupted_paraphrases_total, n_paraphrases_avail)

    poem_pool = rng.sample(range(n_poems_avail), k=n_poems_needed)
    poem_intact_ids = set(poem_pool[:n_true_poems_intact])
    poem_corrupt_ids = set(poem_pool[n_true_poems_intact:])

    fake_pool = rng.sample(range(n_paraphrases_avail), k=n_paraphrases_needed)
    fake_intact_ids = set(fake_pool[:n_paraphrases_used])
    fake_corrupt_ids = set(fake_pool[n_paraphrases_used:])

    print(f"\n[COMPOSIZIONE BILANCIATA DATASET]")
    print(f" ├─ Poesie disponibili:      {n_poems_avail} | Parafrasi disponibili: {n_paraphrases_avail}")
    print(f" ├─ Campioni Totali Target:  ~{max_total_samples}")
    print(f" ├─ Poesie Integre (1.0):    {n_true_poems_intact}")
    print(f" ├─ Parafrasi Integre (0.25):{n_paraphrases_used}")
    print(f" ├─ Poesie da corrompere:    {n_corrupted_poems_total} ({n_corrupted_poems_per_rate} × {n_active_rates} rate)")
    print(f" ├─ Parafrasi da corrompere: {n_corrupted_paraphrases_total} ({n_corrupted_paraphrases_per_rate} × {n_active_rates} rate)")
    print(f" └─ Hard Negatives (0.0):    {n_hard_negatives_total} (REALI {hard_negatives_target_pct*100:.1f}% sul totale)")

    # ---------------- 3. SPLIT PER POESIA ORIGINALE ----------------
    poem_pool_shuffled = poem_pool.copy()
    rng.shuffle(poem_pool_shuffled)
    split_p = int(len(poem_pool_shuffled) * train_split_ratio)
    train_poem_ids = set(poem_pool_shuffled[:split_p])
    test_poem_ids = set(poem_pool_shuffled[split_p:])

    fake_pool_shuffled = fake_pool.copy()
    rng.shuffle(fake_pool_shuffled)
    split_f = int(len(fake_pool_shuffled) * train_split_ratio)
    train_fake_ids = set(fake_pool_shuffled[:split_f])
    test_fake_ids = set(fake_pool_shuffled[split_f:])

    # ---------------- 4. GENERAZIONE INTEGRI ED HARD NEGATIVES ----------------
    intact_train, intact_test = [], []

    # A. Poesie integre (target 1.0)
    for i in poem_intact_ids:
        target = nonLinearLabelling(0.0, rate=poem_rate, floor=poem_floor, ceil=poem_ceil)
        text_raw = true_poems_raw[i].strip()
        if not text_raw.endswith("[VERSE]"):
            text_raw += " [VERSE]"

        if i in train_poem_ids:
            text = dropControlTokens(text_raw, word_tok, dropout_prob=control_dropout_prob)
            intact_train.append((text, target))
        else:
            intact_test.append((text_raw, target))

    # B. Parafrasi integre (target paraphrase_ceil = 0.25) con [VERSE] integrato
    for j in fake_intact_ids:
        target = nonLinearLabelling(0.0, rate=paraphrase_rate, floor=paraphrase_floor, ceil=paraphrase_ceil)
        text_raw = fake_poems_raw[j].strip()
        if not text_raw.endswith("[VERSE]"):
            text_raw += " [VERSE]"

        if j in train_fake_ids:
            text = dropControlTokens(text_raw, word_tok, dropout_prob=control_dropout_prob)
            intact_train.append((text, target))
        else:
            intact_test.append((text_raw, target))

    # C. HARD NEGATIVES (Target 0.0): Anti-Reward Hacking (con [VERSE] sempre presente)
    all_hard_negatives = []
    n_control_negatives = n_hard_negatives_total - num_nonsense_negatives

    # C.1. Sequenze di soli Control Tokens
    for text in controlTokenGeneration(
        max_token_per_sequence=max_control_tokens_per_sequence,
        num_sequences=n_control_negatives,
        seed=seed,
    ):
        text_str = text.strip()
        if not text_str.endswith("[VERSE]"):
            text_str += " [VERSE]"
        all_hard_negatives.append((text_str, 0.0))

    # C.2. Nonsense / Stocastici / Attacchi Mirati tipo XXXXX (Target 0.0)
    sample_tokens_junk = list(word_tok.vocab.values())[:300]
    
    for _ in range(num_nonsense_negatives):
        strategy = rng.choice(["random_subwords", "repeated_token"])
        
        if strategy == "repeated_token":
            # Token subword ripetuti (es. token 88 / caratteri isolati)
            junk_id = rng.choice(sample_tokens_junk)
            junk_toks = [junk_id] * rng.randint(4, 14)
            junk_text = word_tok.detokenize(junk_toks, as_text=True, control_tokens=False)
        
        elif strategy == "mask_attack":
            # Sequenze di mascheramento tipo XXXXX / stringhe anomale
            x_pattern = rng.choice(["XXXXX", "XXXXX XXXXX", "Nel XXXXX del XXXXX vita"])
            junk_text = f"{x_pattern}"
            
        else:
            # Token casuali dal vocabolario
            junk_toks = [rng.choice(sample_tokens_junk) for _ in range(rng.randint(6, 16))]
            junk_text = word_tok.detokenize(junk_toks, as_text=True, control_tokens=False)

        # Garantisce l'apposizione del tag [VERSE] per impedire la scorciatoia di classificazione
        junk_text = junk_text.strip() + " [VERSE]"
        all_hard_negatives.append((junk_text, 0.0))

    # C.3. Ripartizione Train / Test degli Hard Negatives (90/10)
    rng.shuffle(all_hard_negatives)
    split_hn = int(len(all_hard_negatives) * train_split_ratio)
    
    intact_train.extend(all_hard_negatives[:split_hn])
    intact_test.extend(all_hard_negatives[split_hn:])

    random.shuffle(intact_train)
    random.shuffle(intact_test)

    # ---------------- 5. CORROTTI PER RATE ----------------
    rate_datasets = []

    for corr_rate in active_rates:
        corruption_jobs, corruption_meta = [], []

        poem_subset = rng.sample(sorted(poem_corrupt_ids), k=min(n_corrupted_poems_per_rate, len(poem_corrupt_ids)))
        for i in poem_subset:
            target = nonLinearLabelling(corr_rate, rate=poem_rate, floor=poem_floor, ceil=poem_ceil)
            method = WordAndSyllableCorruptionMethod(
                word_tokenizer=word_tok, syl_tokenizer=syl_tok,
                word_percentage=corr_rate, syl_percentage=corr_rate, as_text=True
            )
            corruption_jobs.append((true_poems_raw[i], method))
            corruption_meta.append(("poem", i, i in train_poem_ids, target))

        fake_subset = rng.sample(sorted(fake_corrupt_ids), k=min(n_corrupted_paraphrases_per_rate, len(fake_corrupt_ids)))
        for j in fake_subset:
            target = nonLinearLabelling(corr_rate, rate=paraphrase_rate, floor=paraphrase_floor, ceil=paraphrase_ceil)
            method = WordAndSyllableCorruptionMethod(
                word_tokenizer=word_tok, syl_tokenizer=syl_tok,
                word_percentage=corr_rate, syl_percentage=corr_rate, as_text=True
            )
            corruption_jobs.append((fake_poems_raw[j], method))
            corruption_meta.append(("paraphrase", j, j in train_fake_ids, target))

        if num_workers > 1:
            corrupted_texts = parallelized_corruption_generation(
                corruption_jobs, num_workers=num_workers,
                word_tokenizer_cls=type(word_tok), syl_tokenizer_cls=type(syl_tok)
            )
        else:
            corrupted_texts = sequential_corruption_generation(corruption_jobs)

        rate_train, rate_test = [], []
        for (kind, _src, is_train, target), text in zip(corruption_meta, corrupted_texts):
            text_str = text.strip()
            if not text_str.endswith("[VERSE]"):
                text_str += " [VERSE]"

            if is_train:
                text_aug = dropControlTokens(text_str, word_tok, dropout_prob=control_dropout_prob)
                rate_train.append((text_aug, target))
            else:
                rate_test.append((text_str, target))

        rate_datasets.append({
            "corruption_rate": corr_rate,
            "train_corrupted": rate_train,
            "test_corrupted": rate_test,
        })

    return {
        "intact_train": intact_train,
        "intact_test": intact_test,
        "rate_datasets": rate_datasets,
        "metadata": {
            "max_total_samples": max_total_samples,
            "n_hard_negatives": n_hard_negatives_total,
            "active_rates": active_rates,
        },
    }

In [5]:
import os
import numpy as np
from scipy.stats import spearmanr
import torch
import torch.nn as nn
from typing import Any, Dict, List


def evaluate(
    classifier: nn.Module,
    embedder: nn.Module,
    test_loader: torch.utils.data.DataLoader,
    layers_to_collect: List[int],
    criterion: nn.Module,
    device: torch.device,
    tag: str = "EVAL",
) -> Dict[str, float]:
    """Evaluates the multi-layer classifier and embedder models on continuous target data.

    Directly extracts hidden representations across specified model layers without external collectors, 
    calculating average evaluation loss and binary accuracy at a 0.5 decision threshold.

    Args:
        classifier (nn.Module): Classification neural network accepting multi-layer state dicts.
        embedder (nn.Module): Feature extraction network producing intermediate hidden layer representations.
        test_loader (torch.utils.data.DataLoader): DataLoader supplying validation/testing evaluation batches.
        layers_to_collect (List[int]): Indices of hidden layers to fetch from embedder.
        criterion (nn.Module): Loss function module (e.g., MSELoss, BCEWithLogitsLoss).
        device (torch.device): PyTorch compute target device ('cuda' or 'cpu').
        tag (str): Logging identifier tag for printed evaluation metrics. Defaults to "EVAL".

    Returns:
        Dict[str, float]: Evaluation results dictionary containing:
            - 'loss': Average loss over all evaluation samples.
            - 'accuracy': Binary decision accuracy computed at threshold 0.5.
    """
    classifier.eval()
    embedder.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0
    n_samples = 0
    correct_at_05 = 0

    # Disable gradient tracking during evaluation evaluation phase
    with torch.no_grad():
        for batch in test_loader:
            # Transfer input batch tensors to target device asynchronously
            # input_ids: Shape (batch_size, seq_len)
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            # padding_mask: Shape (batch_size, seq_len)
            padding_mask = batch["padding_mask"].to(device, non_blocking=True)
            # Reshape 1D targets to 2D column vector: Shape (batch_size,) -> (batch_size, 1)
            labels = batch["labels"].to(device).float().unsqueeze(1)

            # 1. Forward pass through embedder to populate internal layer states
            embedder(input_ids, padding_mask=padding_mask)
            # Collect layer state tensors into dictionary mapping: {layer_idx: Tensor(batch_size, seq_len, hidden_dim)}
            layer_outputs = {l: embedder.get_layer_output(l) for l in layers_to_collect}

            # 2. Forward pass through classifier using mixed precision (AMP) matching training setup
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                # preds: Shape (batch_size, 1)
                preds = classifier(layer_outputs, padding_mask=padding_mask)
                # Compute batch loss tensor: Scalar
                loss = criterion(preds, labels)

            # Multiply scalar average batch loss by batch size to track total cumulative loss
            total_loss += loss.item() * labels.size(0)

            # 3. Compute thresholded binary accuracy (threshold >= 0.5)
            # pred_bin & true_bin: Shape (batch_size, 1) float tensors (0.0 or 1.0)
            pred_bin = (preds >= 0.5).float()
            true_bin = (labels >= 0.5).float()
            correct_at_05 += (pred_bin == true_bin).sum().item()

            # Detach predictions and labels from graph and store as 2D NumPy arrays: Shape (batch_size, 1)
            all_preds.append(preds.detach().cpu().numpy())
            all_labels.append(labels.detach().cpu().numpy())
            n_samples += labels.size(0)

    # Concatenate batch arrays along axis 0 and flatten to 1D vectors: Shape (total_samples,)
    preds = np.concatenate(all_preds, axis=0).flatten()
    labels = np.concatenate(all_labels, axis=0).flatten()

    loss_avg = total_loss / max(n_samples, 1)
    acc = correct_at_05 / max(n_samples, 1)

    print(
        f"[{tag}] Loss: {loss_avg:.4f} | "
        f"Accuracy: {acc*100:5.2f}% | "
    )

    return {
        "loss": loss_avg,
        "accuracy": acc,
    }

In [6]:
import torch.nn.functional as F
def group_balanced_mse(pred: torch.Tensor, target: torch.Tensor, num_buckets: int = 5) -> torch.Tensor:
    """
    Computes a group-balanced Mean Squared Error (MSE) 
    loss by dividing the target values into specified buckets.
    """
    pred = pred.squeeze()
    target = target.squeeze()
    
    # Divide i target in bucket di valore
    bucket_limits = torch.linspace(0.0, 1.0, steps=num_buckets + 1, device=target.device)
    bucket_losses = []

    for i in range(num_buckets):
        low, high = bucket_limits[i], bucket_limits[i+1]
        mask = (target >= low) & (target <= high if i == num_buckets - 1 else target < high)
        
        if mask.sum() > 0:
            bucket_loss = F.mse_loss(pred[mask], target[mask])
            bucket_losses.append(bucket_loss)

    return torch.stack(bucket_losses).mean()

def train_classifier(
    classifier,
    embedder,
    train_loader,
    test_loader,
    epochs,
    max_lr,
    save_path,
    device,
    layers_to_collect,
    tag="classifier",
):
    criterion = group_balanced_mse
    optimizer = torch.optim.AdamW(classifier.parameters(), lr=max_lr, weight_decay=0.01)
    
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        steps_per_epoch=len(train_loader),
        epochs=epochs,
        pct_start=0.1,
    )
    
    # GradScaler per Automatic Mixed Precision (AMP)
    scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

    classifier.to(device)
    embedder.to(device)
    embedder.eval()

    start_epoch = 1
    best_test_loss = float("inf") # Inizializzato a infinito per la loss
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    if os.path.exists(save_path):
        try:
            checkpoint = torch.load(save_path, map_location=device)
            if "classifier_state_dict" in checkpoint:
                classifier.load_state_dict(checkpoint["classifier_state_dict"])
            else:
                classifier.load_state_dict(checkpoint)
            
            if "optimizer_state_dict" in checkpoint:
                optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            if "scheduler_state_dict" in checkpoint:
                scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
                
            start_epoch = checkpoint.get("epoch", 0) + 1
            best_test_loss = checkpoint.get("best_test_loss", float("inf"))
            print(f"[{tag}] Ripreso da epoca {start_epoch}/{epochs} (Best Test Loss: {best_test_loss:.4f})")
        except Exception as e:
            print(f"[{tag}] Checkpoint incompatibile, riparto da zero: {e}")
            start_epoch = 1

    if start_epoch > epochs:
        print(f"[{tag}] Training già completato.")
        return True

    for epoch in range(start_epoch, epochs + 1):
        classifier.train()
        total_train_loss = 0.0

        for batch_idx, batch in enumerate(train_loader, 1):
            if batch_idx % 50 == 0:
                print(f"[{tag}] batch {batch_idx}/{len(train_loader)}", end="\r")

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            padding_mask = batch["padding_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device).float().unsqueeze(1)

            optimizer.zero_grad(set_to_none=True)

            # Estrazione feature disconnessa dall'Autograd per l'embedder
            with torch.no_grad():
                embedder(input_ids, padding_mask=padding_mask)
                layer_outputs = {l: embedder.get_layer_output(l).detach() for l in layers_to_collect}

            # Forward pass in AMP (Mixed Precision)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                outputs = classifier(layer_outputs, padding_mask=padding_mask)
                loss = criterion(outputs, labels)

            # Backward pass scalato per AMP
            scaler.scale(loss).backward()

            # Unscale temporaneo prima del clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)

            # Step dell'optimizer e dello scheduler
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # Validation alla fine dell'epoca
        eval_results = evaluate(
            classifier=classifier,
            embedder=embedder,
            test_loader=test_loader,
            layers_to_collect=layers_to_collect,
            criterion=criterion,
            device=device,
            tag=f"{tag}-EVAL",
            print_table=True,
        )

        print(
            f"[{tag}] Epoch [{epoch:02d}/{epochs:02d}] | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Test Loss: {eval_results['loss']:.4f} | "
            f"Accuracy: {eval_results['accuracy']*100:.2f}% | "
        )

        # SALVATAGGIO GUIDATO DALLA TEST LOSS (Non dall'Accuratezza)
        current_test_loss = eval_results['loss']
        if current_test_loss < best_test_loss:
            best_test_loss = current_test_loss
            checkpoint = {
                "epoch": epoch,
                "classifier_state_dict": classifier.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_accuracy": eval_results['accuracy'],
                "best_test_loss": best_test_loss,
                "layers": list(layers_to_collect),
            }
            torch.save(checkpoint, save_path)
            print(f"[{tag}] -> CHECKPOINT salvato (New Best Test Loss: {best_test_loss:.4f})")

    return epoch >= epochs

In [10]:
result = generate_dataset(
    true_poems_raw=true_poems_raw,
    fake_poems_raw=fake_poems_raw,
    word_tok=word_tok,
    syl_tok=syl_tok,
    corruption_rates=[0.2],

    # Dataset ridotto a 15k elementi totali con il 20% reale di target 0.0
    max_total_samples=10000,
    hard_negatives_target_pct=0.20,
    intact_poems_pct=0.12,
    intact_paraphrases_pct=0.15,
    num_nonsense_negatives=1200,

    # Parametri etichette
    poem_floor=0.0,
    poem_ceil=1.0,
    paraphrase_floor=0.0,
    paraphrase_ceil=0.25,

    num_workers=8,
    seed=SEED,
)
# ---- Composizione del dataset congiunto ----
all_train = result["intact_train"].copy()
all_test = result["intact_test"].copy()

for ds in result["rate_datasets"]:
    all_train.extend(ds["train_corrupted"])
    all_test.extend(ds["test_corrupted"])

random.shuffle(all_train)
random.shuffle(all_test)

print(f"\n[DATASET CONGIUNTO]")
print(f" ├─ Train: {len(all_train)}")
print(f" └─ Test:  {len(all_test)}")

# Distribuzione dei target
from collections import Counter
target_dist = Counter(round(t, 3) for _, t in all_train)
total = len(all_train)
print("\n[TARGET] Distribuzione (train):")
for tgt, n in sorted(target_dist.items(), reverse=True):
    print(f"  target={tgt:.3f}: {n} ({100*n/total:.1f}%)")




[COMPOSIZIONE BILANCIATA DATASET]
 ├─ Poesie disponibili:      25333 | Parafrasi disponibili: 10517
 ├─ Campioni Totali Target:  ~10000
 ├─ Poesie Integre (1.0):    1200
 ├─ Parafrasi Integre (0.25):1500
 ├─ Poesie da corrompere:    3975 (3975 × 1 rate)
 ├─ Parafrasi da corrompere: 1325 (1325 × 1 rate)
 └─ Hard Negatives (0.0):    2000 (REALI 20.0% sul totale)
[INFO] Starting parallel corruption on 5300 items with 8 workers (word_tok=WordPieceTokenizer, syl_tok=SyllableTokenizer)...
[SUCCESS] Parallel corruption completed! 5300 items processed.

[DATASET CONGIUNTO]
 ├─ Train: 8999
 └─ Test:  1001

[TARGET] Distribuzione (train):
  target=1.000: 1085 (12.1%)
  target=0.670: 3572 (39.7%)
  target=0.250: 1338 (14.9%)
  target=0.168: 1204 (13.4%)
  target=0.000: 1800 (20.0%)


In [12]:
############################ SYLLABLE CLASSIFIER TRAINING ############################
from models.Pooling import AttentionPoemClassifier


torch.cuda.empty_cache()
DOMAIN = "SYLLABLE_CLASSIFIER"
syl_tok = SyllableTokenizer.from_config()
syl_embedder = PoetEmbedder.from_config(
    tokenizer=syl_tok,
    WEIGHT_PATH=SYLLABLE_TRANSFORMER_PATH,
    device=device,
).to(device)
syl_embedder.eval()
for p in syl_embedder.parameters():
    p.requires_grad = False

syl_classifier = AttentionPoemClassifier(
    embed_dim=EMBEDDING_DIM,
    dropout=DROPOUT,
    device=device,
    attention_hidden=EMBEDDING_DIM,
).to(device)

if os.path.exists(SYL_CLASSIFIER_WEIGHTS_PATH):
    try:
        syl_classifier.load_state_dict_from_path(SYL_CLASSIFIER_WEIGHTS_PATH, device=str(device))
        print(f"[INFO] Caricato checkpoint esistente")
    except Exception as e:
        print(f"[WARN] Checkpoint incompatibile, riparto da zero: {e}")




training_set = RobustAdversarialDataset(all_train, tokenizer=syl_tok, max_len=MAX_SEQ_LEN, domain=DOMAIN)
test_set = RobustAdversarialDataset(all_test, tokenizer=syl_tok, max_len=MAX_SEQ_LEN, domain=DOMAIN)

train_loader = DataLoader(
    training_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, prefetch_factor=2,
    persistent_workers=True,
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

train_classifier(
    classifier=syl_classifier,
    embedder=syl_embedder,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=EPOCHS,
    max_lr=MAX_LR,
    save_path=SYL_CLASSIFIER_WEIGHTS_PATH,
    device=device,
    layers_to_collect=SYL_LAYERS,
    tag="SYL",
)    

[INFO] Caricato modello pre-addestrato da './weights/syllable_transformer_weights.pt'
[WARN] Checkpoint incompatibile, riparto da zero: 'AttentionPoemClassifier' object has no attribute 'load_state_dict_from_path'
[INFO] Tokenizing and chunking for domain 'SYLLABLE_CLASSIFIER'...
[SUCCESS] Pre-allocated 20261 chunks in RAM.
[INFO] Tokenizing and chunking for domain 'SYLLABLE_CLASSIFIER'...
[SUCCESS] Pre-allocated 2335 chunks in RAM.
[SYL] Ripreso da epoca 7/15 (Best Test Loss: 0.0146)


KeyboardInterrupt: 

In [ ]:
############################ WORDPIECE CLASSIFIER TRAINING ############################
torch.cuda.empty_cache()
DOMAIN = "WORDPIECE_CLASSIFIER"
word_tok = WordPieceTokenizer.from_config()
word_embedder = PoetEmbedder.from_config(
    tokenizer=word_tok,
    WEIGHT_PATH=WORDPIECE_TRANSFORMER_PATH,
    device=device,
).to(device)
word_embedder.eval()
for p in word_embedder.parameters():
    p.requires_grad = False

word_classifier = AttentionPoemClassifier.from_config(
    device=device,
    weight_path=WORD_CLASSIFIER_WEIGHTS_PATH,
    num_layers=6,
).to(device)

training_set = RobustAdversarialDataset(all_train, tokenizer=word_tok, max_len=MAX_SEQ_LEN, domain=DOMAIN)
test_set = RobustAdversarialDataset(all_test, tokenizer=word_tok, max_len=MAX_SEQ_LEN, domain=DOMAIN)

train_loader = DataLoader(
    training_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, pin_memory=True, prefetch_factor=2,
    persistent_workers=True,
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

train_classifier(
    classifier=word_classifier,
    embedder=word_embedder,
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=EPOCHS,
    max_lr=MAX_LR,
    save_path=WORD_CLASSIFIER_WEIGHTS_PATH,
    device=device,
    layers_to_collect=WP_LAYERS,
    tag="WORDPIECE",
)    